# Prompts 101: Understanding the Basics

Before optimizing prompts, you need the building blocks: tokens, pricing, quotas, and latency. These fundamentals determine how much you pay, how fast responses arrive, and how much capacity you can use — and every later optimization is measured against them.

## Learning Objectives

By the end of this notebook, you will be able to:
- Count tokens using the Bedrock `CountTokens` API
- Calculate inference costs based on token usage, and see how savings compound at scale
- Understand TPM/RPM quotas and the output burndown rate, and their impact on throughput
- Measure and interpret latency metrics (TTFT, TPS, TTLT, E2E)
- Use the Converse API on `bedrock-runtime`, and recognize when to reach for the Bedrock Mantle endpoint

## Why This Matters

At production scale, small inefficiencies compound dramatically:
- **1,000 extra tokens per request × 1 million requests = 1 billion unnecessary tokens processed** — real money
- Token economics is the foundation for every cost optimization that follows
- Right-sizing `max_tokens` and understanding quotas prevents throttling and protects concurrency

**Duration**: ~30 minutes

## Prerequisites

Before running this notebook, ensure you have:
1. An AWS account with Amazon Bedrock access enabled
2. AWS credentials configured (via `.env` file, AWS CLI, or IAM role)

In [ ]:
# Dependencies are pre-installed from requirements.txt (boto3, python-dotenv,
# anthropic[bedrock] for the Bedrock Mantle section, and the rest).
# Running locally instead? From the repo root:  uv pip install -r requirements.txt
from __future__ import annotations

In [ ]:
# Standard library imports
import os
import time

import boto3

# Third-party imports
from dotenv import load_dotenv

# Local imports - utility functions for pricing calculations
from utils import (
    OUTPUT_BURNDOWN_RATE,
    calculate_actual_cost,
    calculate_tpm_actual,
    calculate_tpm_reservation,
    compare_optimization,
    print_pricing_table,
)

# Load environment variables from .env file
load_dotenv()

# Initialize AWS clients
REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)
bedrock = boto3.client("bedrock", region_name=REGION)
service_quotas = boto3.client("service-quotas", region_name=REGION)

# Model configuration - using Claude Sonnet 4.6 with global CRIS profile
MODEL_ID = "global.anthropic.claude-sonnet-4-6"

print(f"Region: {REGION}")
print(f"Model: {MODEL_ID}")
print(f"boto3 version: {boto3.__version__}")

<div class="alert alert-block alert-info">
<b>Note:</b> This notebook uses Claude Sonnet 4.6 with the <b>global</b> CRIS (Cross-Region Inference Service) profile. The global profile offers ~10% cost savings and higher availability by automatically routing requests to regions with capacity.
</div>

---

## 1. Understanding Tokens

Tokens are the fundamental units that LLMs use to process text. Understanding tokenization is crucial because:
- **Pricing** is based on token count, not character count
- **Context limits** are measured in tokens
- **Quota limits** (TPM) are based on tokens processed

### What is a Token?

A token can be:
- A complete word ("hello" = 1 token)
- Part of a word ("tokenization" might be 2-3 tokens)
- Punctuation or whitespace
- Special characters

**Rule of thumb**: For English text, 1 token is approximately 4 characters or 0.75 words.

### CountTokens API

Amazon Bedrock provides a `CountTokens` API that allows you to count tokens before making inference calls. This is useful for:
- Estimating costs before processing
- Ensuring prompts fit within context limits
- Validating caching strategies (minimum token requirements)

<div class="alert alert-block alert-info">
<b>Note:</b> The CountTokens API is free to use and does not count against your quotas.
</div>

In [ ]:
def count_tokens(text, model_id="anthropic.claude-sonnet-4-6"):
    """
    Count tokens for a given text using Bedrock's CountTokens API.

    Args:
        text: The text to count tokens for
        model_id: The model ID to use for tokenization

    Returns:
        dict with token count and character count
    """
    response = bedrock_runtime.count_tokens(
        modelId=model_id, input={"converse": {"messages": [{"role": "user", "content": [{"text": text}]}]}}
    )

    return {
        "tokens": response["inputTokens"],
        "characters": len(text),
        "chars_per_token": len(text) / response["inputTokens"] if response["inputTokens"] > 0 else 0,
    }


# Test with different prompt lengths
sample_prompts = [
    "Hello, world!",
    "Explain quantum computing in simple terms.",
    "Write a detailed analysis of the economic impact of artificial intelligence on the global workforce over the next decade, including specific sectors that will be most affected and potential mitigation strategies.",
]

print("=" * 70)
print("TOKEN COUNT ANALYSIS")
print("=" * 70)
print(f"{'Prompt Preview':<45} {'Chars':>8} {'Tokens':>8} {'Ratio':>8}")
print("-" * 70)

for prompt in sample_prompts:
    result = count_tokens(prompt)
    preview = prompt[:42] + "..." if len(prompt) > 45 else prompt
    print(f"{preview:<45} {result['characters']:>8} {result['tokens']:>8} {result['chars_per_token']:>7.1f}")

print("=" * 70)

<div class="alert alert-block alert-warning">
<b>Key Insight:</b> English text averages approximately 4 characters per token. However, this ratio varies based on vocabulary complexity, special characters, and formatting. Always use the CountTokens API for accurate counts when estimating costs for production workloads.
</div>

<div class="alert alert-block alert-info">
<b>Token Overhead:</b> The actual token count billed may be slightly higher than just your text content. The underlying API adds a small overhead for:
<ul>
<li>Message formatting (role markers, turn separators)</li>
<li>System prompt wrapper tokens</li>
<li>Tool definitions (if using tools)</li>
</ul>
</div>

---

## 2. Understanding Pricing

Amazon Bedrock uses a pay-per-token pricing model for on-demand mode. Understanding this model is essential for cost optimization.

### Token Pricing Components

| Token Type | Description | Typical Cost Ratio |
|------------|-------------|--------------------|
| Input Tokens | Tokens in your prompt | 1x (base rate) |
| Output Tokens | Tokens generated by model | 3-5x input rate |
| Cache Write | Tokens written to cache | 1.25x input rate |
| Cache Read | Tokens read from cache | 0.1x input rate |

**Key observation**: Output tokens cost significantly more than input tokens. This means:
1. Optimizing prompt length saves money
2. Controlling output length (via `max_tokens`) manages costs
3. Caching can provide substantial savings (up to 90% on cached reads)

In [ ]:
# Display pricing table from utils.py
# Pricing is defined in utils.py for reuse across notebooks
# Always verify current pricing at: https://aws.amazon.com/bedrock/pricing/

print_pricing_table()

### Interactive Cost Calculator

Let's build a cost calculator to understand the economics of token optimization at scale.

In [ ]:
# Scenario: Customer support chatbot
# - Unoptimized: 5,000 token prompt (verbose system instructions + policies)
# - Optimized: 3,000 token prompt (streamlined instructions)
# - Average output: 150 tokens per response

comparison = compare_optimization(
    original_tokens=5000,
    optimized_tokens=3000,
    output_tokens=150,
    num_requests=1_000_000,  # 1 million requests
    model_id=MODEL_ID,
)

print("=" * 70)
print("COST COMPARISON: Token Optimization at Scale")
print("=" * 70)
print("Scenario: 1 million customer support requests")
print(f"Model: {comparison['original']['model']}")
print()

print("Unoptimized (5,000 input tokens per request):")
print(f"  Input cost:  ${comparison['original']['input_cost']:>12,.2f}")
print(f"  Output cost: ${comparison['original']['output_cost']:>12,.2f}")
print(f"  Total cost:  ${comparison['original']['total_cost']:>12,.2f}")

print()
print("Optimized (3,000 input tokens per request):")
print(f"  Input cost:  ${comparison['optimized']['input_cost']:>12,.2f}")
print(f"  Output cost: ${comparison['optimized']['output_cost']:>12,.2f}")
print(f"  Total cost:  ${comparison['optimized']['total_cost']:>12,.2f}")

print()
print(f"SAVINGS: ${comparison['savings']:,.2f} ({comparison['savings_pct']:.1f}% reduction)")
print("=" * 70)

### ROI at Different Scales

Let's see how optimization savings compound at different request volumes.

In [ ]:
def roi_at_scale(original_tokens, optimized_tokens, output_tokens=150, model_id=MODEL_ID):
    """
    Calculate ROI of prompt optimization at different scales.
    """
    scales = [
        (1_000, "1K (pilot)"),
        (10_000, "10K (small)"),
        (100_000, "100K (medium)"),
        (1_000_000, "1M (large)"),
        (10_000_000, "10M (enterprise)"),
    ]

    print("=" * 80)
    print(f"ROI ANALYSIS: {original_tokens} -> {optimized_tokens} tokens")
    print("=" * 80)
    print(f"{'Scale':<20} {'Original':>15} {'Optimized':>15} {'Savings':>15} {'%':>10}")
    print("-" * 80)

    for num_requests, label in scales:
        result = compare_optimization(original_tokens, optimized_tokens, output_tokens, num_requests, model_id)
        print(
            f"{label:<20} ${result['original']['total_cost']:>13,.2f} ${result['optimized']['total_cost']:>13,.2f} ${result['savings']:>13,.2f} {result['savings_pct']:>9.1f}%"
        )

    print("=" * 80)


# Run ROI analysis
roi_at_scale(original_tokens=5000, optimized_tokens=3000)

<div class="alert alert-block alert-warning">
<b>Key Insight:</b> A 40% reduction in input tokens (5,000 to 3,000) yields approximately 34% total cost savings. At enterprise scale (10M requests), this translates to $60,000+ in savings - and this is before applying prompt caching, which can save up to 90% on cached read tokens!
</div>

---

## 3. Latency Impact

Token count affects not just cost but also latency. Understanding this relationship helps you build responsive applications.

### Key Latency Metrics

| Metric | What It Measures | Why It Matters |
|--------|------------------|----------------|
| **TTFT** (Time to First Token) | How quickly the model starts responding | Critical for perceived speed - users see something happening |
| **TPS** (Tokens per Second) | How fast text appears after it starts | Affects how quickly the full response is generated |
| **TTLT** (Time to Last Token) | Total time until the last token is generated | The actual generation time (excludes app overhead) |
| **E2E Latency** | Total time from request to complete response | The full picture including your application overhead |

### Measuring Real API Latency

Let's measure actual latency from the Bedrock API using the **ConverseStream API** to capture real TTFT and TPS. The streaming response lets us time exactly when the first token arrives, giving accurate TTFT measurement rather than an estimate.

In [ ]:
def measure_latency(prompt, model_id=MODEL_ID, max_tokens=200, stream_output=False):
    """
    Measure latency metrics using the ConverseStream API.

    Uses streaming to measure real TTFT (time until first content delta),
    TPS (output tokens per second), TTLT (server-side latency), and E2E (total round-trip).

    Args:
        prompt: The user message
        model_id: Model to use
        max_tokens: Maximum tokens in response
        stream_output: If True, print tokens as they arrive
    """
    start_time = time.time()

    response = bedrock_runtime.converse_stream(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": 0},
    )

    stream = response["stream"]
    first_token_time = None
    output_text = ""
    output_tokens = 0
    input_tokens = 0
    latency_ms = None

    for event in stream:
        if "contentBlockDelta" in event:
            if first_token_time is None:
                first_token_time = time.time()
            delta = event["contentBlockDelta"]["delta"]
            if "text" in delta:
                output_text += delta["text"]
                if stream_output:
                    print(delta["text"], end="", flush=True)
        elif "metadata" in event:
            usage = event["metadata"].get("usage", {})
            input_tokens = usage.get("inputTokens", 0)
            output_tokens = usage.get("outputTokens", 0)
            latency_ms = event["metadata"].get("metrics", {}).get("latencyMs")

    if stream_output:
        print()  # newline after streaming

    end_time = time.time()

    ttft_ms = (first_token_time - start_time) * 1000 if first_token_time else 0
    e2e_ms = (end_time - start_time) * 1000  # Full client round-trip
    ttlt_ms = latency_ms if latency_ms is not None else e2e_ms  # Server-side model latency
    generation_time_s = end_time - first_token_time if first_token_time else 0
    tps = output_tokens / generation_time_s if generation_time_s > 0 else 0

    return {
        "ttft_ms": ttft_ms,
        "ttlt_ms": ttlt_ms,
        "e2e_ms": e2e_ms,
        "tps": tps,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }


# Measure real latency with different prompt sizes
print("=" * 90)
print("REAL API LATENCY MEASUREMENT (ConverseStream)")
print("=" * 90)

test_prompts = [
    ("Short", "What is 2+2? Answer briefly."),
    ("Medium", "Explain photosynthesis in 2-3 sentences."),
    ("Long", "Explain machine learning training in one paragraph."),
]

# Run each prompt with live streaming output
results = []
for label, prompt in test_prompts:
    print(f"\n[{label}] Prompt: {prompt}")
    print(f"  Response: ", end="")
    metrics = measure_latency(prompt, max_tokens=150, stream_output=True)
    results.append((label, metrics))

# Summary metrics table
print(f"\n{'Prompt':<10} {'TTFT':>12} {'TTLT':>12} {'E2E':>12} {'TPS':>10} {'In':>8} {'Out':>8}")
print("-" * 90)

for label, metrics in results:
    print(
        f"{label:<10} {metrics['ttft_ms']:>10.0f}ms {metrics['ttlt_ms']:>10.0f}ms {metrics['e2e_ms']:>10.0f}ms {metrics['tps']:>8.1f}/s {metrics['input_tokens']:>8} {metrics['output_tokens']:>8}"
    )

print("=" * 90)
print("\nKey metrics:")
print("- TTFT (Time to First Token): Time from request start to first content delta (includes network)")
print("- TTLT (Time to Last Token): Server-side model latency (excludes app/network overhead)")
print("- E2E (End-to-End): Total client round-trip from request start to stream end")
print("- TPS: Output tokens generated per second during generation phase")

<div class="alert alert-block alert-info">
<b>Note:</b> We use <code>converse_stream()</code> above to measure <b>real TTFT</b> by timing when the first content delta arrives. The non-streaming <code>converse()</code> API only returns the complete response, so TTFT cannot be directly measured - you'd have to estimate it. Streaming also enables better UX: users see tokens appear as they're generated, making the response feel faster even if total generation time (TTLT) is the same. TPS (Tokens per Second) indicates how fast the model generates output after starting.
</div>

---

## 4. Bedrock API endpoints

Bedrock exposes Claude through **two endpoints**. They share the same AWS SigV4 auth and run on the **same next-generation inference engine**, so the model behaves identically on both — you choose by the **API shape** your code wants, not as a default. This workshop runs entirely on **Converse over `bedrock-runtime`**; the Bedrock Mantle section below shows the other endpoint so you recognize the option.

```
bedrock-runtime.{region}.amazonaws.com     →  Converse API   ·  InvokeModel API
bedrock-mantle.{region}.api.aws            →  Anthropic Messages API   ·   OpenAI Responses / Chat Completions
```

### `bedrock-runtime` — Converse and InvokeModel

**Converse** is the unified API: one request/response shape across every Bedrock provider (Anthropic, Amazon, Meta, Mistral…), so switching models is a one-line change. **InvokeModel** is the provider-native path on the *same* endpoint — for Claude its body *is* the Anthropic Messages format (`anthropic_version: "bedrock-2023-05-31"`). Both are current; Converse is `bedrock-runtime`-only.

| Aspect | InvokeModel | **Converse** (workshop default) |
|--------|-------------|--------------------------------|
| Interface | Per-provider request format | Unified across all models |
| Code portability | Different code per provider | Same code for Claude, Nova, Mistral… |
| Multi-turn | Manual history management | Built-in conversation history |
| Tool use | Provider-specific | Native unified support |
| Prompt caching | Provider-specific | `cachePoint` checkpoints |

### Which `bedrock-runtime` API should you use?

| Scenario | Use |
|----------|-----|
| New application, multi-turn chat, tool use, caching — **anything in this workshop** | **Converse** |
| You need a **provider-native request field** Converse doesn't surface, without leaving `bedrock-runtime` | **InvokeModel** |
| Existing InvokeModel code | Keep it — both are current |

> **For this workshop**: every notebook uses the **Converse API**. The cell below makes the call you'll use everywhere. The second endpoint, `bedrock-mantle` (native Anthropic Messages API), is covered in the Bedrock Mantle section below.

In [ ]:
def converse(prompt, model_id=MODEL_ID, max_tokens=100, temperature=0):
    """
    Make a simple inference call using the Converse API.

    Args:
        prompt: The user message
        model_id: Model to use for inference
        max_tokens: Maximum tokens in the response
        temperature: Sampling temperature (0 = deterministic)

    Returns:
        dict with response text and usage metrics
    """
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": temperature},
    )

    return {
        "text": response["output"]["message"]["content"][0]["text"],
        "stop_reason": response["stopReason"],
        "usage": response["usage"],
        "latency_ms": response.get("metrics", {}).get("latencyMs", None),
    }


# Make a simple inference call
result = converse("What is the capital of France? Answer in one sentence.")

print("=" * 70)
print("CONVERSE API DEMO")
print("=" * 70)
print(f"Response: {result['text']}")
print(f"Stop Reason: {result['stop_reason']}")
print("=" * 70)

### Understanding Usage Metrics

Every Converse API response includes usage metrics that are essential for cost tracking and optimization.

In [ ]:
# Display detailed usage metrics from the previous call
usage = result["usage"]

print("=" * 70)
print("USAGE METRICS BREAKDOWN")
print("=" * 70)
print()
print("Token Counts:")
print(f"  inputTokens:           {usage.get('inputTokens', 0):>8}  (uncached input)")
print(f"  outputTokens:          {usage.get('outputTokens', 0):>8}  (generated response)")
print(f"  cacheReadInputTokens:  {usage.get('cacheReadInputTokens', 0):>8}  (read from 5m cache - 0.1x cost)")
print(f"  cacheWriteInputTokens: {usage.get('cacheWriteInputTokens', 0):>8}  (written to 5m cache - 1.25x cost)")
print(
    f"  totalTokens:           {usage.get('totalTokens', usage.get('inputTokens', 0) + usage.get('outputTokens', 0)):>8}  (total processed)"
)
print()

# Calculate actual cost using the utility function
actual_cost = calculate_actual_cost(
    input_tokens=usage.get("inputTokens", 0), output_tokens=usage.get("outputTokens", 0), model_id=MODEL_ID
)

print(f"Estimated Cost: ${actual_cost:.8f}")
if result.get("latency_ms"):
    print(f"Latency: {result['latency_ms']} ms")
print("=" * 70)

### Usage Metrics Reference

| Metric | Description | Cost Impact |
|--------|-------------|-------------|
| `inputTokens` | Tokens in prompt that were NOT cached | 1x base rate |
| `outputTokens` | Tokens generated by model | 3-5x base rate |
| `cacheWriteInputTokens` | Tokens written to 5m cache (first request) | 1.25x base rate |
| `cacheReadInputTokens` | Tokens read from 5m cache (subsequent requests) | 0.1x base rate (90% savings!) |
| `totalTokens` | Sum of all tokens processed | Varies by type |

---

## 5. Bedrock Mantle

So far you've called Bedrock through the **Converse API** on the `bedrock-runtime` endpoint. There's a second endpoint — **`bedrock-mantle`** — that exposes provider-native APIs (Anthropic's Messages API, OpenAI-compatible Chat Completions, Anthropic's Responses API) directly through Bedrock's IAM/SigV4 auth.

It runs alongside `bedrock-runtime`, not instead of it. You pick whichever endpoint exposes the feature you need.


### Why Mantle exists

Mantle exposes **provider-native API shapes** directly through Bedrock's IAM/SigV4 auth. Reach for it when you specifically want one of these — not as a default:

| Reason to use Mantle | What it gives you |
|---|---|
| **Native wire format** | Keep code already written against the Anthropic Messages API or the OpenAI SDK — just change the base URL. |
| **OpenAI-compatible surfaces** | Responses + Chat Completions APIs, including stateful conversations via the Responses API. |
| **Separate quota pool** | Mantle traffic is counted separately from your `bedrock-runtime` quota, so heavy Mantle workloads don't compete with your Converse calls. |

### Endpoint comparison

| | `bedrock-runtime` | `bedrock-mantle` |
|---|---|---|
| **APIs** | Converse / ConverseStream, InvokeModel | Anthropic Messages (`/anthropic/v1/messages`); OpenAI Responses + Chat Completions (`/v1`) |
| **Auth** | IAM SigV4 | IAM SigV4 **or** Bedrock API key / bearer |
| **Model IDs** | CRIS profiles — `global.`/`us.`/`eu.` prefixes | In-region only — bare `anthropic.claude-…` |
| **Quotas** | Single combined TPM (5× output burndown, Claude 3.7+) | Separate input-TPM + output-TPM, no RPM; cached input exempt |
| **Batch / Provisioned / inference profiles** | ✅ | runtime-only |

The two endpoints share one next-generation inference engine, so the model behaves identically — pick by API shape, not as a default. Opus 4.7+ runs on it regardless of endpoint, so code already on `bedrock-runtime` needs no change to use the latest models.

### Which API should I use?

| Your situation | Use |
|---|---|
| New code, multi-provider, multi-turn, tool use, caching — anything in this workshop | **Converse** (`bedrock-runtime`) |
| You need a **provider-native request field** Converse doesn't expose, on the standard runtime endpoint (CRIS pricing, batch/provisioned) | **InvokeModel** (`bedrock-runtime`) |
| You're **porting Anthropic-SDK or OpenAI-SDK code**, need the OpenAI-compatible surface or a separate quota pool — and in-region inference is acceptable (no CRIS) | **Mantle** (`bedrock-mantle`) |

### Model ID format on Mantle

Mantle uses **un-prefixed, in-region** IDs (no `global.`/`us.` — there's no cross-region inference), and it's **per-model** — not every model is served on it.

| Model | `bedrock-runtime` | `bedrock-mantle` |
|---|---|---|
| Haiku 4.5 | `global.anthropic.claude-haiku-4-5-20251001-v1:0` | `anthropic.claude-haiku-4-5` |
| Opus 4.8 | `global.anthropic.claude-opus-4-8` | `anthropic.claude-opus-4-8` |

The demo below runs the **same prompt and same model (Haiku 4.5)** through both endpoints so you can compare the request/response shapes directly — Converse returns a Bedrock-unified envelope, Mantle returns Anthropic's native Messages object, while the underlying inference is identical.

In [ ]:
# Bedrock Mantle uses the anthropic[bedrock] SDK (pinned in requirements.txt).
try:
    from anthropic import AnthropicBedrockMantle
    MANTLE_AVAILABLE = True
    print("AnthropicBedrockMantle imported. Ready to call Mantle.")
except ImportError:
    MANTLE_AVAILABLE = False
    print("anthropic[bedrock] not installed — the remaining Mantle cells will skip themselves cleanly.")

# Mantle endpoint configuration (region-specific; no CRIS prefix).
MANTLE_REGION = "us-east-1"
MANTLE_BASE_URL = f"https://bedrock-mantle.{MANTLE_REGION}.api.aws"
CONVERSE_HAIKU = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
MANTLE_HAIKU = "anthropic.claude-haiku-4-5"

### Calling Mantle: same query, two endpoints

Run the same prompt through Converse and through Mantle. The response shapes differ — Converse returns a Bedrock-unified envelope, Mantle returns Anthropic's native Messages object — but the underlying inference is the same.

In [ ]:
# 1) Same prompt through Converse, on Haiku 4.5 (the model we'll also call on Mantle)
PROMPT = "List three reasons why prompt caching reduces cost. One sentence each."

converse_resp = bedrock_runtime.converse(
    modelId=CONVERSE_HAIKU,  # global.anthropic.claude-haiku-4-5-20251001-v1:0
    messages=[{"role": "user", "content": [{"text": PROMPT}]}],
    inferenceConfig={"maxTokens": 200, "temperature": 0},
)

print("=== Converse (bedrock-runtime, Haiku 4.5) ===")
print(converse_resp["output"]["message"]["content"][0]["text"])
print(f"\nUsage: input={converse_resp['usage']['inputTokens']}, output={converse_resp['usage']['outputTokens']}")

In [ ]:
# 2) Same prompt, same model (Haiku 4.5), through Mantle (Anthropic Messages API)
if MANTLE_AVAILABLE:
    mantle = AnthropicBedrockMantle(
        aws_region=MANTLE_REGION,
        base_url=f"{MANTLE_BASE_URL}/anthropic",
    )

    mantle_resp = mantle.messages.create(
        model=MANTLE_HAIKU,  # anthropic.claude-haiku-4-5 — same model as the Converse call above
        max_tokens=200,
        messages=[{"role": "user", "content": PROMPT}],
    )

    print("=== Mantle (bedrock-mantle, Haiku 4.5, Messages API) ===")
    print(mantle_resp.content[0].text)
    print(
        f"\nUsage: input={mantle_resp.usage.input_tokens}, "
        f"output={mantle_resp.usage.output_tokens}"
    )
else:
    print("Skipped — install anthropic[bedrock] to run this cell.")

### Takeaways

- **Two endpoints, same auth.** `bedrock-runtime` (Converse) for unified cross-provider code; `bedrock-mantle` for Anthropic-native / OpenAI-compatible wire formats and a separate quota pool.
- **Mantle uses un-prefixed, in-region IDs** — `anthropic.claude-haiku-4-5`, not `global.anthropic.claude-haiku-4-5-20251001-v1:0` — and bills at the in-region (non-discounted) rate. It's also per-model: Haiku 4.5 and Opus 4.8 are served, Sonnet 4.6 is not.
- **`anthropic[bedrock]` SDK handles SigV4** — you don't sign requests by hand.

For this workshop you stay on the **Converse API** throughout; Mantle is here so you recognize the option when you meet it in production.

---

## 6. TPM and RPM Quotas

Amazon Bedrock enforces throughput limits to ensure fair resource allocation. Understanding these quotas is essential for building reliable production systems.

### Quota Types

| Quota | Description |
|-------|-------------|
| **TPM** (Tokens Per Minute) | Maximum tokens processed per minute |
| **RPM** (Requests Per Minute) | Maximum API calls per minute |

Both limits apply simultaneously - you must stay within both to avoid throttling.

<div class="alert alert-block alert-warning">
<b>Important:</b> TPM quota consumption is calculated as: <code>InputTokenCount + CacheWriteInputTokens + (OutputTokenCount x output_burndown_rate)</code>. For <b>Claude 3.7 Sonnet and newer</b> (Sonnet 4.6, Haiku 4.5, Opus 4.8) the output burndown rate is <b>5x</b>, meaning each output token consumes 5 tokens of your TPM quota; older/standard models burn at 1x. This is because output generation is more compute-intensive than input processing.
</div>

### TPM Quota: Before vs After Request

Let's see how TPM quota consumption works in practice:
1. **Before request**: Bedrock reserves quota based on `max_tokens` setting
2. **After request**: Actual consumption is calculated from real token usage

This is important because over-setting `max_tokens` reserves more quota than needed, reducing concurrency.

In [ ]:
# TPM Calculation: Before vs After Request
# Using Claude Sonnet 4.6 with 5x output burndown rate
# Functions imported from utils.py: calculate_tpm_reservation, calculate_tpm_actual, calculate_actual_cost

# Make a real API call to demonstrate
print("=" * 80)
print(f"TPM QUOTA: BEFORE vs AFTER REQUEST (Claude Sonnet 4.6, {OUTPUT_BURNDOWN_RATE}x burndown)")
print("=" * 80)

# Set up request parameters
test_prompt = "What are three benefits of cloud computing? Be brief."
max_tokens_setting = 500  # What we set in the API call

# Count input tokens first
token_count = count_tokens(test_prompt)
input_tokens = token_count["tokens"]

# BEFORE: Calculate reservation
tpm_reserved = calculate_tpm_reservation(input_tokens, max_tokens_setting)

print("\n>>> BEFORE REQUEST (Quota Reservation)")
print(f"   Input tokens:           {input_tokens:>8}")
print(f"   max_tokens setting:     {max_tokens_setting:>8}")
print(
    f"   Output quota reserved:  {max_tokens_setting * OUTPUT_BURNDOWN_RATE:>8}  ({max_tokens_setting} x {OUTPUT_BURNDOWN_RATE})"
)
print("   ------------------------------------")
print(f"   TOTAL TPM RESERVED:     {tpm_reserved:>8}")

# Make the actual API call
result = converse(test_prompt, max_tokens=max_tokens_setting)
usage = result["usage"]
actual_output = usage["outputTokens"]
actual_input = usage["inputTokens"]

# AFTER: Calculate actual consumption
tpm_actual = calculate_tpm_actual(actual_input, actual_output)
actual_cost = calculate_actual_cost(actual_input, actual_output, MODEL_ID)

print("\n<<< AFTER REQUEST (Actual Usage)")
print(f"   Input tokens:           {actual_input:>8}")
print(f"   Output tokens:          {actual_output:>8}")
print(
    f"   Output TPM consumed:    {actual_output * OUTPUT_BURNDOWN_RATE:>8}  ({actual_output} x {OUTPUT_BURNDOWN_RATE})"
)
print("   ------------------------------------")
print(f"   ACTUAL TPM CONSUMED:    {tpm_actual:>8}")

# Show the difference
tpm_wasted = tpm_reserved - tpm_actual
efficiency = (tpm_actual / tpm_reserved) * 100 if tpm_reserved > 0 else 100

print("\n=== ANALYSIS")
print(f"   TPM reserved:           {tpm_reserved:>8}")
print(f"   TPM actually used:      {tpm_actual:>8}")
print(f"   TPM over-reserved:      {tpm_wasted:>8}  (wasted quota capacity)")
print(f"   Quota efficiency:       {efficiency:>7.1f}%")
print(f"\n$$$ ACTUAL COST: ${actual_cost:.6f}")
print("=" * 80)

<div class="alert alert-block alert-warning">
<b>Key Insight:</b> The output burndown rate is <b>5x for Claude 3.7 Sonnet and newer</b> (including Sonnet 4.6, Haiku 4.5, and Opus 4.8) — each output token consumes 5 tokens of your TPM quota; older/standard models burn at 1x. <b>But remember:</b>
<ul>
<li><b>TPM affects concurrency, not cost</b> - you're billed for actual tokens, not reserved quota</li>
<li><b>Over-setting max_tokens wastes quota capacity</b> - the example above shows how setting max_tokens=500 but only using ~100 tokens reserves 5x more quota than needed</li>
<li><b>Right-size max_tokens</b> - set it to expected output + 10-15% buffer, not the model maximum</li>
</ul>
</div>

### Additional Quota Considerations

- **In-region vs CRIS**: Cross-Region Inference Service (CRIS) has separate quotas from in-region calls
- **max_tokens reservation**: Setting `max_tokens` reserves that capacity during request processing
- **Quota increases**: Request increases via the AWS Service Quotas console for production workloads
- **Concurrent requests**: Both TPM and RPM limits apply simultaneously

---

## Summary

In this notebook, you learned the fundamental concepts for working with LLMs on Amazon Bedrock:

| Concept | Key Takeaway |
|---------|-------------|
| **Tokens** | ~4 characters per token for English; use `CountTokens` API for accurate counts; watch for API overhead |
| **Pricing** | Output tokens cost 3-5x more than input; 5m cache reads save 90%; global CRIS saves ~10% |
| **Cost at Scale** | 40% token reduction = 34% cost savings; compounds to $60K+ at 10M requests |
| **Latency** | TTFT critical for UX; TTLT = total generation time; use streaming for responsiveness |
| **Endpoints** | Two: `bedrock-runtime` (Converse — workshop default — + InvokeModel) and `bedrock-mantle` (native Anthropic Messages API). Same auth, same engine; pick by API shape. |
| **Usage Metrics** | Track `inputTokens`, `outputTokens`, and cache metrics for cost visibility |
| **Quotas** | TPM formula includes output burndown (5x); TPM affects concurrency, not cost |

### What's Next

Next, head to **02-langfuse-observability.ipynb** to instrument these calls. Optimization techniques (model selection, prompt design, parameter tuning, caching, and more) live in the **Optimization Playbook** (`../02-optimization-playbook/`), where you'll learn:
- Model selection strategies (right-sizing for your use case)
- Prompt design best practices (clear instructions, few-shot examples)
- Parameter tuning (`max_tokens`, `stop_sequences`)
- Prompt caching fundamentals
- Adaptive thinking with `effort`
- LLM routing, RAG, compression, conversation & memory, and more

---

## Additional Resources

- [Amazon Bedrock Pricing](https://aws.amazon.com/bedrock/pricing/)
- [Amazon Bedrock Quotas](https://docs.aws.amazon.com/bedrock/latest/userguide/quotas.html)
- [Converse API Documentation](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html)
- [Prompt Caching Documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html)